# Portfolio / Brokers — Dev Playground

Interactive notebook for exercising the `/portfolio/*` API surface end-to-end. Two paths:

1. **In-process** — call the FastAPI app via `TestClient` (no server needed). Fastest iteration.
2. **Live HTTP** — point at a running `pdm run dev` server. Use this when validating CORS, frontend wiring, or hitting real broker APIs (Angel One, Zerodha, Groww, Wint Wealth).

Toggle by setting `MODE` below.

**New in this revision:** five out of six sources are now `kind=API` and pull holdings without a CSV upload. CSV upload remains as a manual fallback for every source.

| Slug | Kind | Auth path |
|---|---|---|
| `zerodha` | API | Web-login enctoken (ZERODHA_USER_ID / PASSWORD / TOTP_SECRET) |
| `zerodha-coin` | API | Reuses Zerodha session |
| `groww` | API | Reverse-engineered web API (GROWW_USER_ID / PASSWORD / optional TOTP) |
| `angel-one` | API | Official SmartAPI (ANGEL_ONE_API_KEY / CLIENT_CODE / PASSWORD / TOTP_SECRET) |
| `wint-wealth` | API | OTP-bound (WINT_USER_ID + interactive `/start-login` + `/otp`) |
| `dezerv` | CSV | No public API |


In [1]:
MODE = "in_process"  # or "http"
BASE = "http://localhost:8000/api/v1"

from pathlib import Path
FIXTURES = Path.cwd().parent / "tests" / "fixtures" / "broker_csvs"

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""

def get(path, **kw):
    r = client.get(f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text

def post(path, **kw):
    r = client.post(f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text

print("Mode:", MODE)

Mode: in_process


/Users/arpitarya/my_programs/alpha-forge/backend/app/routes/market.py:66: FastAPIDeprecationWarning: `regex` has been deprecated, please use `pattern` instead
  interval: str = Query("1d", regex="^(1m|5m|15m|1h|1d|1w|1M)$"),
/Users/arpitarya/my_programs/alpha-forge/backend/app/routes/market.py:67: FastAPIDeprecationWarning: `regex` has been deprecated, please use `pattern` instead
  period: str = Query("1y", regex="^(1d|5d|1M|3M|6M|1y|5y|max)$"),


## 1. List configured sources

Six sources: 5 API (Zerodha, Coin, Groww, Angel One, Wint Wealth) + 1 CSV (Dezerv). Status reflects whether the required env vars are present.

In [2]:
import json
_, body = get("/portfolio/sources")
for s in body["sources"]:
    print(f"  {s['slug']:14} {s['kind']:4} {s['status']:13} {s['label']}")

  zerodha        api  unconfigured  Zerodha (Kite)
  zerodha-coin   api  unconfigured  Zerodha Coin
  groww          api  unconfigured  Groww
  angel-one      api  unconfigured  Angel One (SmartAPI)
  wint-wealth    api  unconfigured  Wint Wealth
  dezerv         csv  unconfigured  Dezerv


## 2. Sync a single API source

Triggers the source's HTTP login + holdings fetch. Cached enctoken/JWT lives under `.cache/brokers/<slug>.json` so subsequent syncs skip re-login.

Picks one source by name — change `SLUG` to try others (`zerodha-coin`, `groww`, `angel-one`).

In [6]:
SLUG = "zerodha"
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status)
print(json.dumps(body, indent=2)[:800])

Sync failed for zerodha
Traceback (most recent call last):
  File "/Users/arpitarya/my_programs/alpha-forge/backend/app/routes/portfolio.py", line 133, in sync_source
    holdings = await src.sync()
               ^^^^^^^^^^^^^^^^
  File "/Users/arpitarya/my_programs/alpha-forge/backend/app/services/brokers/base.py", line 108, in sync
    holdings = await self.fetch()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/arpitarya/my_programs/alpha-forge/backend/app/services/brokers/zerodha_kite.py", line 146, in fetch
    enctoken = await acquire_enctoken()
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arpitarya/my_programs/alpha-forge/backend/app/services/brokers/zerodha_kite.py", line 71, in acquire_enctoken
    raise RuntimeError(f"Zerodha credentials missing: {', '.join(missing)}")
RuntimeError: Zerodha credentials missing: ZERODHA_USER_ID, ZERODHA_PASSWORD, ZERODHA_TOTP_SECRET


400
{
  "detail": "Sync failed: Zerodha credentials missing: ZERODHA_USER_ID, ZERODHA_PASSWORD, ZERODHA_TOTP_SECRET"
}


## 3. Sync all API sources at once

Fan-out across every `kind=API` source. Returns a per-source success/error map plus aggregate totals. Use this from the frontend's "Refresh All" button.

In [ ]:
status, body = post("/portfolio/sources/sync-all")
print(status)
for slug, res in body.get("results", {}).items():
    if res.get("ok"):
        print(f"  ✓ {slug:14} {res['count']} holdings")
    else:
        print(f"  ✗ {slug:14} {res.get('error', '')[:80]}")
print("\nTotals:", body.get("totals"))

## 4. Wint Wealth — OTP-bound login

Two-step flow because OTPs land on the user's phone/email:

1. `POST /sources/wint-wealth/start-login` triggers the OTP send.
2. `POST /sources/wint-wealth/otp` with the received code → backend caches the JWT.
3. `POST /sources/wint-wealth/sync` works for ~30 days afterwards.

Run only against a real backend (`MODE="http"`); the cells below are sketches you fill in interactively.

In [ ]:
# Step 1: trigger OTP
# status, body = post("/portfolio/sources/wint-wealth/start-login")
# print(status, body)

In [ ]:
# Step 2: submit OTP from your phone/email
# OTP = "123456"
# status, body = post("/portfolio/sources/wint-wealth/otp", json={"code": OTP})
# print(status, body)

In [ ]:
# Step 3: sync holdings using the cached JWT
# status, body = post("/portfolio/sources/wint-wealth/sync")
# print(status, body)

## 5. CSV upload (manual fallback)

Every source — including the API ones — accepts a CSV upload. Useful when:
- you don't want to store broker creds locally,
- the upstream API is throttling/blocking, or
- you're working off a historical export.

API sources delegate to the CSV parser of the same broker.

In [ ]:
FIXTURES_MAP = {
    "zerodha":      "zerodha_holdings.csv",
    "zerodha-coin": "zerodha_coin_holdings.csv",
    "groww":        "groww_stocks.csv",
    "dezerv":       "dezerv_holdings.csv",
    "wint-wealth":  "wint_wealth_holdings.csv",
}
for slug, fname in FIXTURES_MAP.items():
    path = FIXTURES / fname
    if not path.exists():
        print(f"  {slug:14} → (no fixture {fname})")
        continue
    with path.open("rb") as f:
        status, body = post(f"/portfolio/sources/{slug}/upload", files={"file": (fname, f, "text/csv")})
    if status == 200:
        print(f"  {slug:14} → {status} ({body.get('holdings_count', '?')} holdings)")
    else:
        print(f"  {slug:14} → {status} {body}")

## 6. Aggregate view

After sync/upload, all sources merge into one portfolio. Allocation is by `asset_class`.

In [ ]:
_, body = get("/portfolio/holdings")
print("Totals:", body["totals"])
print("\nAllocation:")
for s in body["allocation"]:
    print(f"  {s['asset_class']:12} ₹{s['value']:>14,.0f}  ({s['pct']:>5.1f}%)")
print("\nFirst 3 holdings:")
for h in body["holdings"][:3]:
    print("  ", h["source"], h["symbol"], h["current_value"])

## 7. Filter by source

Same endpoints accept `?source=<slug>` to scope the view to one broker.

In [ ]:
_, body = get("/portfolio/holdings", params={"source": "zerodha"})
print("Zerodha-only totals:", body["totals"])
for h in body["holdings"][:5]:
    print("  ", h["symbol"], h["quantity"], h["current_value"])

## 8. Treemap layout

Pre-computed squarified layout — frontend just absolute-positions cells with the supplied `left/top/width/height` percentages.

In [ ]:
_, body = get("/portfolio/treemap")
for c in body["cells"][:8]:
    print(f"  {c['symbol']:14} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 9. Rebalance suggestions

Drift = actual - target. Default targets are 60/15/15/5/3/2 (equity/MF/bond/gold/crypto/cash). Suggestions fire above ±5%.

In [ ]:
_, body = get("/portfolio/rebalance")
print("Drift:")
for d in body["drift"]:
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body["suggestions"]:
    print("  -", s["action"])

## 10. Inspect cached session tokens

API sources persist tokens (Zerodha enctoken, Groww access_token, Wint JWT) under `.cache/brokers/<slug>.json` so daily syncs skip the login dance. Delete a file to force a fresh login next sync.

In [ ]:
import os
cache_dir = Path(os.getenv("BROKER_CACHE_DIR", ".cache/brokers")).resolve()
if cache_dir.exists():
    for f in sorted(cache_dir.glob("*.json")):
        print(f"  {f.name:24} {f.stat().st_size:>5}B  mtime={f.stat().st_mtime:.0f}")
else:
    print("  (no cache yet — run a sync)")

## 11. Reset state

Clears the in-memory cache for one source so you can re-sync or re-upload.

In [ ]:
for slug in ["zerodha", "zerodha-coin", "groww", "angel-one", "wint-wealth", "dezerv"]:
    post(f"/portfolio/sources/{slug}/reset")
_, body = get("/portfolio/sources")
for s in body["sources"]:
    print(f"  {s['slug']:14} {s['status']:13} ({s['holdings_count']} holdings)")